# ⚡ Pipeline 4: LightGBM with 21-Day Recency Profiles
### Recency-Weighted Target Aggregations · 7-Day Holdout CV · Adversarial Domain Adaptation · 3-Seed Bagging

**Key Pipeline Architecture:**
1. **Base Features**: Cyclical hour/day-of-week, holiday markers (Thanksgiving/Christmas/NYE), weather deviations, gas price ratio, multi-label amenities, power-per-port.
2. **Hierarchical + Recency Profiles**: Global station/hour/weekend target encoding + **recent 21-day station-hour utilization profiles** (`st_hr_mean_recent21`, `st_hr_wk_mean_recent21`).
3. **7-Day Holdout Split (Nov 18–24)**: Leak-safe out-of-time validation to accurately estimate generalization on unseen future timeframes.
4. **Adversarial Validation**: Stripped calendar features to compute winter shift sample weights for the test set.
5. **3-Seed Bagging Ensemble**: Retraining on 100% data across seeds `[42, 100, 2024]` with adversarial weighting.

## 1. Imports & Data Loading

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load datasets
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

print(f'Train dataset shape: {train.shape}')
print(f'Test dataset shape:  {test.shape}')
train.head(3)

## 2. Base Feature Engineering & 21-Day Recency Profiles

In [ ]:
def engineer_base_features(df_train, df_test):
    df_all = pd.concat([df_train, df_test], ignore_index=True)
    df_all['dt'] = pd.to_datetime(df_all['timestamp'])

    # Time Features
    df_all['hour']       = df_all['dt'].dt.hour
    df_all['minute']     = df_all['dt'].dt.minute
    df_all['time_step']  = df_all['hour'] + df_all['minute'] / 60.0
    df_all['dayofweek']  = df_all['dt'].dt.dayofweek
    df_all['is_weekend'] = (df_all['dayofweek'] >= 5).astype(int)
    df_all['month']      = df_all['dt'].dt.month
    df_all['weekofyear'] = df_all['dt'].dt.isocalendar().week.astype(int)

    # Cyclical encodings
    df_all['sin_hour'] = np.sin(2 * np.pi * df_all['time_step'] / 24.0)
    df_all['cos_hour'] = np.cos(2 * np.pi * df_all['time_step'] / 24.0)
    df_all['sin_dow']  = np.sin(2 * np.pi * df_all['dayofweek'] / 7.0)
    df_all['cos_dow']  = np.cos(2 * np.pi * df_all['dayofweek'] / 7.0)

    # Holiday indicators
    df_all['is_thanksgiving_week'] = ((df_all['month'] == 11) & (df_all['dt'].dt.day >= 24) & (df_all['dt'].dt.day <= 30)).astype(int)
    df_all['is_christmas_week']    = ((df_all['month'] == 12) & (df_all['dt'].dt.day >= 20) & (df_all['dt'].dt.day <= 26)).astype(int)
    df_all['is_nye']               = ((df_all['month'] == 12) & (df_all['dt'].dt.day >= 29)).astype(int)

    # Weather imputation & deviation
    df_all['temperature_f']    = df_all['temperature_f'].fillna(df_all.groupby(['city', 'month'])['temperature_f'].transform('median'))
    df_all['precipitation_mm'] = df_all['precipitation_mm'].fillna(0.0)
    df_all['is_raining']       = (df_all['precipitation_mm'] > 0).astype(int)

    city_hr_temp = df_all.groupby(['city', 'hour'])['temperature_f'].transform('mean')
    df_all['temp_diff_city_hr']    = df_all['temperature_f'] - city_hr_temp
    city_gas                       = df_all.groupby('city')['gas_price_per_gallon'].transform('mean')
    df_all['gas_price_ratio_city'] = df_all['gas_price_per_gallon'] / city_gas

    # Amenities multi-label flags
    for am, nm in [('WiFi','wifi'),('Restroom','restroom'),('Shopping Mall','shopping_mall'),
                   ('Park','park'),('Fast Food','fast_food'),('Hotel','hotel'),
                   ('Convenience Store','convenience_store'),('Grocery Store','grocery_store')]:
        df_all['has_' + nm] = df_all['amenities_nearby'].fillna('').str.contains(am).astype(int)
    df_all['num_amenities']  = df_all['amenities_nearby'].fillna('').apply(lambda x: len([i for i in x.split(', ') if i]))
    df_all['power_per_port'] = df_all['power_output_kw'] / df_all['ports_total'].replace(0, 1)

    cat_cols = ['station_id','network','city','state','location_type','charger_type','pricing_type','weather_condition','local_event']
    for c in cat_cols:
        cats = df_all[c].dropna().astype(str).unique()
        df_all[c] = df_all[c].astype(pd.CategoricalDtype(categories=cats))

    return df_all.iloc[:len(df_train)].copy(), df_all.iloc[len(df_train):].copy(), cat_cols

def build_hierarchical_features(source_tr, *dfs_to_map):
    def agg(g, **kw): return source_tr.groupby(g, observed=False)['utilization_rate'].agg(**kw).reset_index()
    st_hr_wk = agg(['station_id','hour','is_weekend'], st_hr_wk_mean='mean', st_hr_wk_std='std', st_hr_wk_min='min', st_hr_wk_max='max')
    st_hr    = agg(['station_id','hour'],              st_hr_mean='mean', st_hr_std='std')
    st_base  = agg('station_id',                       st_base_mean='mean', st_base_std='std')
    loc_hr   = agg(['location_type','hour'],            loc_hr_mean='mean')
    net_hr   = agg(['network','hour'],                  net_hr_mean='mean')

    # --- Recency-weighted profiles (last 21 days of source_tr only) ---
    max_dt = source_tr['dt'].max()
    recent_mask = source_tr['dt'] > (max_dt - pd.Timedelta(days=21))
    recent = source_tr[recent_mask]

    st_hr_recent = recent.groupby(['station_id', 'hour'], observed=False)['utilization_rate'].agg(st_hr_mean_recent21='mean').reset_index()
    st_hr_wk_recent = recent.groupby(['station_id', 'hour', 'is_weekend'], observed=False)['utilization_rate'].agg(st_hr_wk_mean_recent21='mean').reset_index()

    def mp(df):
        return df.merge(st_hr_wk,          on=['station_id','hour','is_weekend'], how='left') \
                 .merge(st_hr,             on=['station_id','hour'],               how='left') \
                 .merge(st_base,           on='station_id',                        how='left') \
                 .merge(loc_hr,            on=['location_type','hour'],             how='left') \
                 .merge(net_hr,            on=['network','hour'],                   how='left') \
                 .merge(st_hr_recent,      on=['station_id','hour'],               how='left') \
                 .merge(st_hr_wk_recent,   on=['station_id','hour','is_weekend'],  how='left')

    return (mp(source_tr), *[mp(d) for d in dfs_to_map]) if dfs_to_map else mp(source_tr)

## 3. Prepare Leak-Safe 7-Day Holdout Split (Nov 18–Nov 24)

In [ ]:
print('Building features...')
train_df, test_df, cat_cols = engineer_base_features(train, test)

DROP = ['id', 'timestamp', 'dt', 'station_name', 'amenities_nearby', 'utilization_rate']

df_sorted = train_df.sort_values(['station_id', 'dt']).reset_index(drop=True)
cutoff    = df_sorted['dt'].max() - pd.Timedelta(days=7)
tr_raw    = df_sorted[df_sorted['dt'] <= cutoff].copy()
val_raw   = df_sorted[df_sorted['dt'] >  cutoff].copy()

tr, val = build_hierarchical_features(tr_raw, val_raw)
feat_cols = [c for c in tr.columns if c not in DROP]

print(f'Train set (prior to Nov 18): {len(tr):,} rows')
print(f'Val set (Nov 18–Nov 24):     {len(val):,} rows')
print(f'Total features ({len(feat_cols)}): {feat_cols}')

## 4. LightGBM Validation Training & Evaluation

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.06,
    'num_leaves': 127,
    'max_depth': 10,
    'feature_fraction': 0.80,
    'bagging_fraction': 0.80,
    'bagging_freq': 1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_estimators': 2000,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

print('Training LightGBM on holdout split...')
model_cv = lgb.LGBMRegressor(**params)
model_cv.fit(
    tr[feat_cols], tr['utilization_rate'],
    eval_set=[(tr[feat_cols], tr['utilization_rate']), (val[feat_cols], val['utilization_rate'])],
    eval_names=['Train', 'Val'],
    callbacks=[lgb.early_stopping(40, verbose=100)]
)

pred_val = np.clip(model_cv.predict(val[feat_cols]), 0.02, 0.98)
rmse_7d  = np.sqrt(mean_squared_error(val['utilization_rate'], pred_val))
mae_7d   = mean_absolute_error(val['utilization_rate'], pred_val)
r2_7d    = r2_score(val['utilization_rate'], pred_val)

print('='*50)
print('PIPELINE 4 (7-DAY HOLDOUT NOV 18–24) METRICS:')
print(f'  Best Iteration : {model_cv.best_iteration_}')
print(f'  RMSE           : {rmse_7d:.5f}')
print(f'  MAE            : {mae_7d:.5f}')
print(f'  R²             : {r2_7d:.5f}')
print('='*50)

## 5. Adversarial Validation (Calendar-Stripped)

In [ ]:
ADV_FEATS = ['temperature_f', 'precipitation_mm', 'weather_condition',
             'gas_price_per_gallon', 'hour', 'dayofweek', 'is_raining']

adv_tr = train_df[ADV_FEATS].copy()
adv_te = test_df[ADV_FEATS].copy()
for c in ['weather_condition']:
    adv_tr[c] = adv_tr[c].astype('category')
    adv_te[c] = adv_te[c].astype('category')

adv_combined = pd.concat([
    adv_tr.assign(is_test=0),
    adv_te.assign(is_test=1)
], ignore_index=True)
adv_combined['weather_condition'] = adv_combined['weather_condition'].astype('category')

adv_model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, random_state=42, n_jobs=-1, verbose=-1)
adv_model.fit(adv_combined[ADV_FEATS], adv_combined['is_test'])

adv_auc = roc_auc_score(adv_combined['is_test'], adv_model.predict_proba(adv_combined[ADV_FEATS])[:, 1])
print(f'Adversarial AUC (Calendar-Stripped): {adv_auc:.4f}')

likeness = adv_model.predict_proba(adv_tr[ADV_FEATS])[:, 1]
sample_weights = likeness / likeness.mean()
print(f'Sample Weights: min={sample_weights.min():.3f} | mean={sample_weights.mean():.3f} | max={sample_weights.max():.3f}')

## 6. Full Retraining (100% Data + 3-Seed Ensemble)

In [ ]:
print('Retraining LightGBM on 100% data...')
best_trees = model_cv.best_iteration_ or 1000
full_tr, full_te = build_hierarchical_features(train_df, test_df)
for c in cat_cols:
    if c in full_tr.columns:
        full_tr[c] = full_tr[c].astype('category')
        full_te[c] = full_te[c].astype('category')

X_full, y_full = full_tr[feat_cols], full_tr['utilization_rate']
X_test = full_te[feat_cols]

seeds = [42, 100, 2024]
preds_list = []
for s in seeds:
    m = lgb.LGBMRegressor(**{**params, 'random_state': s, 'n_estimators': best_trees})
    m.fit(X_full, y_full, sample_weight=sample_weights)
    preds_list.append(m.predict(X_test))
    print(f'  ✓ Seed {s} finished training.')

final_preds = np.clip(np.mean(preds_list, axis=0), 0.02, 0.98)

## 7. Feature Importance & Submission Output

In [ ]:
# Top 15 Feature Importances
imp_df = pd.DataFrame({'Feature': feat_cols, 'Importance': m.feature_importances_}).sort_values('Importance', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(imp_df['Feature'].head(15)[::-1], imp_df['Importance'].head(15)[::-1], color='#3b82f6')
plt.title('Top 15 Feature Importances (Pipeline 4 LightGBM)')
plt.xlabel('Split Importance')
plt.tight_layout()
plt.show()

# Save submission
sub_df = pd.DataFrame({'id': test['id'], 'utilization_rate': np.round(final_preds, 4)})
sub_df.to_csv('submission (4).csv', index=False, float_format='%.4f')
print(f'Predictions saved -> submission (4).csv')

if os.path.exists('submission_best.csv'):
    gt = pd.read_csv('submission_best.csv').sort_values('id').reset_index(drop=True)
    sub_eval = sub_df.sort_values('id').reset_index(drop=True)
    lb_score = np.sqrt(mean_squared_error(gt['utilization_rate'], sub_eval['utilization_rate']))
    print(f'Local Leaderboard Benchmark RMSE vs submission_best.csv: {lb_score:.5f}')